In [ ]:
import base64
import json
import pprint
import time

import requests
import requests_aws4auth

print("requests_aws4auth: ", requests_aws4auth.__version__)

In [ ]:
OPENSEARCH_HOST = "https://hogehoge:10004"
OPENSEARCH_USER = "OpenSearchのユーザID"
OPENSEARCH_PASSWORD = "OpenSearchのパスワード"

In [ ]:
REGION = "us-east-1"  # amazon nova multimodal embedding(リージョンはus-east-1のみ)
SERVICE_NAME = "bedrock"
ACCESS_KEY_FOR_BEDROCK = "Amazon Bedrockに触れるIAMユーザのaccessキー"
SECRET_KEY_FOR_BEDROCK = "Amazon Bedrockに触れるIAMユーザのsecretキー"
# https://github.com/tedder/requests-aws4auth/blob/main/requests_aws4auth/aws4auth.py
AUTH_FOR_AWS = requests_aws4auth.AWS4Auth(ACCESS_KEY_FOR_BEDROCK,
                                          SECRET_KEY_FOR_BEDROCK,
                                          REGION,
                                          SERVICE_NAME)
BEDROCK_ENDPOINT = "https://bedrock-runtime.us-east-1.amazonaws.com/model/amazon.nova-2-multimodal-embeddings-v1:0/invoke"  # amazon nova multimodal embedding(リージョンはus-east-1のみ)
JSON_FILE_PATH = "OpenSearchに登録するデータを1つのjsonファイルで纏めたもの"

##### 投入するデータを投入できる形に前処理するための準備

In [ ]:
def read_json_file(json_file_path):
    with open(file=json_file_path, mode="r", encoding="utf-8") as f:
        json_to_dict = json.load(f)
    return json_to_dict

In [ ]:
def get_image_from_url_and_convert_to_base64(image_url):
    image_response = requests.get(url=image_url)
    image_content = image_response.content
    image_base64 = base64.b64encode(s=image_content).decode("utf-8")
    return image_base64

In [ ]:
def text_embedding_using_amazon_nova_multimodal(target_text):
    # https://docs.aws.amazon.com/nova/latest/userguide/embeddings-schema.html
    # https://docs.aws.amazon.com/ja_jp/nova/latest/userguide/complete-request-schema.html
    text_embedding_payload = {"schemaVersion": "nova-multimodal-embed-v1",
                              "taskType": "SINGLE_EMBEDDING",
                              "singleEmbeddingParams": {"embeddingPurpose": "GENERIC_INDEX",
                                                        "embeddingDimension": 1024,
                                                        "text": {"truncationMode": "END",
                                                                 "value": target_text}
                                                       }
                             }
    response = requests.post(url=BEDROCK_ENDPOINT,
                             auth=AUTH_FOR_AWS,
                             headers={"Content-Type": "application/json",
                                      "Accept": "application/json"},
                             json=text_embedding_payload)
    response_dict = response.json()
    return response_dict["embeddings"][0]["embedding"]

In [ ]:
def image_embedding_using_amazon_nova_multimodal(target_image_url):
    # https://docs.aws.amazon.com/nova/latest/userguide/embeddings-schema.html
    # https://docs.aws.amazon.com/ja_jp/nova/latest/userguide/complete-request-schema.html
    image_embedding_payload = {"schemaVersion": "nova-multimodal-embed-v1",
                               "taskType": "SINGLE_EMBEDDING",
                               "singleEmbeddingParams": {"embeddingPurpose": "GENERIC_INDEX",
                                                         "embeddingDimension": 1024,
                                                         "image": {"detailLevel": "DOCUMENT_IMAGE",
                                                                   "format": "jpeg",
                                                                   "source": {"bytes": get_image_from_url_and_convert_to_base64(image_url=target_image_url)}
                                                                  }
                                                        }
                              }
    response = requests.post(url=BEDROCK_ENDPOINT,
                             auth=AUTH_FOR_AWS,
                             headers={"Content-Type": "application/json",
                                      "Accept": "application/json"},
                             json=image_embedding_payload)
    response_dict = response.json()
    return response_dict["embeddings"][0]["embedding"]

In [ ]:
data_dict_list = read_json_file(json_file_path=JSON_FILE_PATH)

In [ ]:
data_dict_list_for_opensearch = []
for data_dict in data_dict_list:
    data_dict_for_opensearch = {}
    data_dict_for_opensearch["product_name"] = data_dict["name"]
    data_dict_for_opensearch["description"] = data_dict["description"]
    data_dict_for_opensearch["features"] = data_dict["detail"]
    data_dict_for_opensearch["image_url"] = data_dict["image_url"]
    data_dict_for_opensearch["page_url"] = data_dict["product_url"]
    data_dict_for_opensearch["product_name_vector"] = text_embedding_using_amazon_nova_multimodal(target_text=data_dict["name"])
    data_dict_for_opensearch["description_vector"] = text_embedding_using_amazon_nova_multimodal(target_text=data_dict["description"])
    data_dict_for_opensearch["image_vector"] = image_embedding_using_amazon_nova_multimodal(target_image_url=data_dict["image_url"])
    data_dict_for_opensearch["features_vector"] = text_embedding_using_amazon_nova_multimodal(target_text=" ".join(data_dict["detail"]))
    data_dict_for_opensearch["sizes"] = data_dict["size"]
    data_dict_for_opensearch["prices"] = data_dict["price"]
    data_dict_list_for_opensearch.append(data_dict_for_opensearch)
    time.sleep(3)

##### データを一括で対象のDB indexに登録
##### https://docs.opensearch.org/latest/api-reference/document-apis/bulk/

In [ ]:
target_index_name = "my-index-name"  # 03_dbindex_control.ipynbで作ったもの # データを登録するindexの名前

In [ ]:
bulk_data_register_url = "{a}/_bulk".format(a=OPENSEARCH_HOST)
register_bulk_data_list = []
register_action_dict = {"index": {"_index": target_index_name}}
# https://docs.opensearch.org/latest/api-reference/document-apis/bulk/#request-body
for data_dict_for_opensearch in data_dict_list_for_opensearch:
    register_bulk_data_list.append(json.dumps(register_action_dict,
                                              ensure_ascii=False))
    register_bulk_data_list.append(json.dumps(data_dict_for_opensearch,
                                              ensure_ascii=False))
register_bulk_data_payload = "\n".join(register_bulk_data_list) + "\n"

In [ ]:
response = requests.post(url=bulk_data_register_url,
                         auth=(OPENSEARCH_USER,
                               OPENSEARCH_PASSWORD),
                         headers={"Content-Type": "application/json"},
                         data=register_bulk_data_payload.encode("utf-8"),
                         verify=False)
print(response.status_code)
pprint.pprint(response.json())

##### 対象のDB indexに登録されているデータを確認
##### https://docs.opensearch.org/latest/api-reference/search-apis/search/

In [ ]:
target_index_name = "my-index-name"  # 03_dbindex_control.ipynbで作ったもの # 登録したデータの中身を確認したいindexの名前

In [ ]:
registered_data_check_url = "{a}/{b}/_search".format(a=OPENSEARCH_HOST, b=target_index_name)

In [ ]:
registered_data_check_payload = {"_source": ["product_name",  # 指定のデータ項目だけを取得
                                             "description",
                                             "features",
                                             "image_url",
                                             "page_url",
                                             "sizes",
                                             "prices"],
                                 "size": 5}  # 対象のインデックスに登録されているデータを5件だけ確認

In [ ]:
response_2 = requests.post(url=registered_data_check_url,
                           auth=(OPENSEARCH_USER,
                                 OPENSEARCH_PASSWORD),
                           headers={"Content-Type": "application/json"},
                           json=registered_data_check_payload,
                           verify=False)
print(response_2.status_code)
pprint.pprint(response_2.json())